# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** M. Ramadhan Titan Dwi Cahya  
**NIM:** 202310370311068  
**Username GitHub:** MRamadhan000  

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tracks_features.parquet'

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions)
untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [ ]:
df_temporal = (
    df.with_columns(pl.col("release_date").str.to_date("%Y-%m-%d", strict=False).alias("date"))
    .sort("date")
    .drop_nulls(["date"])
)

# Contoh 1: Rolling mean untuk fitur energy dengan window size 50 lagu berurutan
df_rolling = df_temporal.with_columns(
    pl.col("energy").rolling_mean(window_size=50).alias("energy_rolling_mean")
)

# Contoh 2: Dynamic grouping per tahun untuk melihat rata-rata tempo (PERBAIKAN DI SINI)
df_yearly = (
    df_temporal.group_by(pl.col("date").dt.year().alias("release_year"))
    .agg([
        pl.col("tempo").mean().alias("avg_tempo"),
        pl.col("danceability").mean().alias("avg_danceability"),
        pl.len().alias("total_tracks")
    ])
    .sort("release_year")
)

print(df_yearly.head())

## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [ ]:
import plotly.express as px

df_yearly = (
    df_temporal.group_by(pl.col("date").dt.year().alias("release_year"))
    .agg([
        pl.col("tempo").mean().alias("avg_tempo"),
        pl.col("danceability").mean().alias("avg_danceability"),
        pl.col("energy").mean().alias("avg_energy"),
        pl.len().alias("total_tracks")
    ])
    .sort("release_year")
)

fig1 = px.line(
    df_yearly.to_pandas(),
    x="release_year",
    y="avg_energy",
    title="<b>Tren Rata-rata Energi Musik dari Tahun ke Tahun</b>",
    labels={
        "release_year": "Tahun Rilis", 
        "avg_energy": "Rata-rata Energi (Energy Score)"
    },
    markers=True,
    template="plotly_dark"
)

fig1.update_xaxes(
    rangeslider_visible=True,
    tickmode="linear",
    dtick=5
)

fig1.update_layout(
    title_font_size=18,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    hovermode="x unified"
)

fig1.show()

In [ ]:
import plotly.express as px
import polars as pl

features_to_plot = ["danceability", "energy", "acousticness"]
nama = {
    "danceability": "Danceability",
    "energy": "Energy",
    "acousticness": "Acousticness",
}

label = {}
total = df.height
for f in features_to_plot:
    q1 = df[f].quantile(0.25, interpolation="linear")
    q3 = df[f].quantile(0.75, interpolation="linear")
    iqr = q3 - q1
    batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = df.filter((pl.col(f) < batas_bawah) | (pl.col(f) > batas_atas)).height
    persen = n_out / total * 100
    label[f] = f"<b>{nama[f]}</b><br>{n_out:,} lagu outlier ({persen:.1f}%)"

df_melted = (
    df.select(features_to_plot)
    .unpivot(variable_name="Feature", value_name="Score")
    .with_columns(pl.col("Feature").replace(label))
    .to_pandas()
)

fig2 = px.box(
    df_melted,
    x="Feature",
    y="Score",
    color="Feature",
    color_discrete_sequence=["#CFD8DC"] * 3,   # kotak abu-abu pucat
    title=(
        "<b>Persebaran Nilai Danceability, Energy, dan Acousticness</b><br>"
        "<span style='font-size:13px; color:#555555;'>"
        "</span>"
    ),
    labels={"Feature": "", "Score": ""},
    template="plotly_white",
)

fig2.update_traces(
    boxpoints="outliers",
    line=dict(color="#78909C", width=1.5),
    marker=dict(
        color="#78909C",
        outliercolor="crimson",
        size=5,
        opacity=0.7,
        line=dict(width=0),
    ),
    hovertemplate="Skor: %{y:.2f}<extra></extra>",
)

fig2.update_layout(
    title_font_size=18,
    title_font_color="black",
    title_x=0.5,
    title_xanchor="center",
    showlegend=False,
    margin=dict(l=50, r=50, t=100, b=70),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
)

fig2.update_xaxes(showgrid=False, tickfont=dict(size=13, color="black"))
fig2.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    zeroline=False,
    tickfont=dict(size=11, color="#777777"),
)

fig2.show()

In [ ]:
import plotly.express as px
import polars as pl

numeric_cols = ["danceability", "energy", "loudness", "speechiness", "acousticness", "valence", "tempo"]

corr_df = df.select(numeric_cols).to_pandas().corr(method="pearson")

fig3 = px.imshow(
    corr_df,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="<b>Matriks Korelasi Heatmap Antar-Variabel Numerik Spotify</b>",
    labels=dict(color="<b>Koefisien Korelasi (r)</b>"),
    template="plotly_white"
)

fig3.update_layout(
    title_font_size=18,
    title_font_color="black",  
    title_x=0.5,               
    title_xanchor="center",    
    xaxis_title_font_size=13,
    yaxis_title_font_size=13,
    margin=dict(l=100, r=80, t=100, b=90),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
)

fig3.update_xaxes(
    side="top", 
    tickfont=dict(size=12, color="black", family="Arial Black"),  
    title_font=dict(size=14, color="black")
)

fig3.update_yaxes(
    tickfont=dict(size=12, color="black", family="Arial Black"),
    title_font=dict(size=14, color="black")
)

fig3.show()

In [ ]:
import plotly.express as px
import polars as pl

top_artists = (
    df.group_by("artists")
    .agg(pl.len().alias("track_count"))
    .sort("track_count", descending=True)
    .head(10)
    .to_pandas()
)

fig4 = px.bar(
    top_artists,
    x="artists",
    y="track_count",
    color="track_count",
    color_continuous_scale="Blues",
    title="<b>Top 10 Artis dengan Jumlah Trek Terbanyak</b>",
    labels={
        "artists": "<b>Nama Artis</b>", 
        "track_count": "<b>Jumlah Trek (Lagu)</b>"
    },
    template="plotly_white"
)

fig4.update_layout(
    title_font_size=18,
    title_font_color="black",
    title_x=0.5,
    title_xanchor="center",
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    showlegend=False,
    height=600,
    margin=dict(l=60, r=50, t=90, b=100), 
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)"
)

fig4.update_xaxes(
    tickangle=-35,
    tickfont=dict(size=11, color="black", family="Arial Black"),
    title_font=dict(size=14, color="black")
)

fig4.update_yaxes(
    showgrid=True,
    gridcolor='rgba(0, 0, 0, 0.1)',
    tickfont=dict(size=11, color="black", family="Arial Black"),
    title_font=dict(size=14, color="black")
)

fig4.show()

In [ ]:
import plotly.express as px
import polars as pl

df_monthly = (
    df.with_columns(
        pl.col("release_date").str.to_date("%Y-%m-%d", strict=False).alias("date")
    )
    .drop_nulls(["date"])
    .with_columns([
        pl.col("date").dt.month().alias("release_month"),
        pl.col("date").dt.strftime("%B").alias("month_name")
    ])
    .group_by(["release_month", "month_name"])
    .agg([
        pl.col("energy").mean().alias("avg_energy"),
        pl.len().alias("total_tracks")
    ])
    .sort("release_month")
)

fig5 = px.bar(
    df_monthly.to_pandas(),
    x="month_name",
    y="total_tracks",
    color="avg_energy",
    color_continuous_scale="Viridis",
    title="<b>Volume Rilisan dan Rata-rata Energi Musik per Bulan</b>",
    labels={
        "month_name": "<b>Bulan Rilis</b>",
        "total_tracks": "<b>Jumlah Total Trek Dirilis</b>",
        "avg_energy": "<b>Rata-rata Energy</b>"
    },
    template="plotly_white"
)

fig5.update_layout(
    title_font_size=18,
    title_font_color="black",
    title_x=0.5,               
    title_xanchor="center",
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    height=600,                
    margin=dict(l=60, r=80, t=90, b=80),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)"
)

fig5.update_xaxes(
    tickangle=-25,
    tickfont=dict(size=11, color="black", family="Arial Black"),
    title_font=dict(size=14, color="black")
)

fig5.update_yaxes(
    showgrid=True,
    gridcolor='rgba(0, 0, 0, 0.1)',
    tickfont=dict(size=11, color="black", family="Arial Black"),
    title_font=dict(size=14, color="black")
)

print(df_monthly.to_pandas().to_string(index=False))

fig5.show()

In [ ]:
import plotly.express as px
import polars as pl

# 1. Ambil sampel data agar ringan
df_sample = df.sample(n=min(2000, df.height), seed=42).to_pandas()

# 2. Membuat Scatter Plot 3D interaktif
fig6 = px.scatter_3d(
    df_sample,
    x="tempo",
    y="energy",
    z="valence",
    color="danceability",
    hover_name="name",
    color_continuous_scale="Viridis",
    title="<b>Analisis 3D Multi-Dimensi: Tempo, Energy, Valence, dan Danceability</b>",
    template="plotly_white"
)

# 3. Mempercantik tata letak, mengatur judul di tengah, serta memperpanjang tinggi grafik (height=700)
fig6.update_layout(
    title_font_size=18,
    title_font_color="black",
    title_x=0.5,
    title_xanchor="center",
    height=700,                # <--- MEMPERPANJANG TINGGI KANVAS GRAFIK
    margin=dict(l=50, r=50, t=90, b=50),
    scene=dict(
        xaxis_title='<b>Tempo (BPM)</b>',
        yaxis_title='<b>Energy</b>',
        zaxis_title='<b>Valence</b>',
        # Anda juga bisa mengatur aspek rasio box 3D jika diperlukan
        aspectmode='cube'      # Membuat volume kubus 3D tampak seimbang dan proporsional
    )
)

# Menampilkan grafik di Jupyter Notebook
fig6.show()

## Bagian 4: Rumusan 5 Insight Analitik Utama

Tuliskan minimal 5 temuan atau wawasan penting yang Anda peroleh dari visualisasi di atas:

1. **Insight 1 (Tren Waktu & Musiman):**  
   Berdasarkan grafik **Tren Rata-rata Energi Musik dari Tahun ke Tahun**, tren tingkat energi musik mengalami perubahan yang dari waktu ke waktu. Pada puluhan tahun pertama hingga tahun 1950an, tingkat energi musik sangat tidak stabil dan naik-turun secara drastis, seperti sempat merosot tajam pada tahun 1925 sebelum akhirnya melonjak tinggi di awal tahun 1950an. Memasuki periode 1960 hingga 1980an, energi musik cenderung stabil di kisaran sedang dengan puncak tertinggi yang sempat mendekati angka 0,6 pada awal tahun 1980-an. Setelah mengalami penurunan ke titik terendah pada awal tahun 1990-an, tingkat energi musik mulai menunjukkan tren peningkatan yang sangat konsisten setiap tahunnya dari pertengahan 1990an hingga akhirnya kembali mencapai titik energi tertinggi di angka hampir 0,6 pada tahun 2020.   

2. **Insight 2 (Distribusi & Anomali):**  
   Berdasarkan grafik **Persebaran Nilai Danceability, Energy, dan Acousticness**, ketiga variabel yaitu Danceability, Energy, dan Acousticness memiliki rentang nilai yang sama dari 0 hingga 1, namun dengan sebaran yang berbeda. Variabel Danceability dan Energy memiliki nilai tengah (median) di sekitar angka 0,5, yang menunjukkan bahwa nilai data dari kedua kategori ini tersebar cukup seimbang di area tengah. Sementara itu, Acousticness memiliki garis tengah yang sedikit lebih rendah di angka sekitar 0,4 dengan kotak sebaran yang lebih melebar ke arah atas. Yang paling menarik, tidak ada satu pun data yang menjadi pencilan atau data ekstrem (outlier) pada ketiga kategori tersebut, yang dibuktikan dengan keterangan 0 lagu outlier (0,0%) serta garis batas (whisker) yang menjangkau nilai terendah 0 dan tertinggi 1 secara merata.   

3. **Insight 3 (Korelasi Kunci):**  
   Berdasarkan **Matriks Korelasi Heatmap Antar-Variabel Numerik Spotify** tersebut, temuan penting pertama menunjukkan adanya hubungan yang sangat erat antara tingkat energi (energy) dan tingkat kenyaringan suara (loudness) dengan angka korelasi sebesar 0,84. Artinya, lagu-lagu yang bersemangat tinggi hampir selalu diproduksi dengan volume atau suara yang lebih keras, yang bisa menjadi panduan penting bagi para pembuat musik dalam meracik tata suara lagu mereka.

   Di sisi lain, terdapat hubungan berbanding terbalik yang sangat kuat antara tingkat energi dengan keakustikan lagu (acousticness) sebesar -0,80. Hal ini berarti lagu-lagu yang menggunakan instrumen akustik murni cenderung mempunyai energi dan kenyaringan suara yang lebih rendah, sangat kontras dengan lagu-lagu modern yang menggunakan produksi elektronik.

   Selain itu, terdapat pula korelasi positif yang cukup kuat antara tingkat kemudahan untuk dipakai berjoget (danceability) dan suasana hati yang ceria (valence) di angka $0,56$. Temuan ini memperlihatkan bahwa lagu-lagu yang asyik untuk berdansa umumnya juga membawa nuansa yang menyenangkan, sehingga sangat cocok untuk dimanfaatkan dalam menyusun daftar putar (playlist) musik penambah suasana hati.

5. **Insight 4 (Perilaku Kategori / Segmen):**  
   Berdasarkan grafik **Volume Rilisan dan Rata-rata Energi Musik per Bulan**, terdapat anomali yang sangat mencolok pada jumlah total trek yang dirilis pada bulan Januari (mencapai 180.164 lagu), yang jauh lebih tinggi dibandingkan bulan-bulan lainnya yang rata-rata berkisar antara 49.000 hingga 105.000 lagu. Hal ini menunjukkan adanya tren musiman di mana awal tahun sering kali menjadi periode paling masif bagi perilisan karya musik baru.

   Meskipun volume rilisan sangat bervariasi dari bulan ke bulan, rata-rata tingkat energi musik relatif stabil dan konsisten berada di kisaran angka 0,5 hingga 0,54 sepanjang tahun. Nilai energi terendah tercatat pada bulan Januari (0,46), sementara tingkat energi tertinggi diraih pada bulan Juli (0,54).

   Bulan dengan jumlah rilisan terbanyak (Januari) justru memiliki rata-rata energi terendah di antara seluruh bulan, sebaliknya bulan dengan volume rilisan paling sedikit (Desember, 49.767 lagu) atau bulan dengan rilisan menengah (Juli) justru menunjukkan tingkat energi rata-rata yang cukup tinggi.

7. **Insight 5 (Rekomendasi Praktis):**  
   Berdasarkan seluruh temuan analisis, disarankan agar para musisi menghindari perilisan lagu berenergi tinggi di awal tahun mengingat persaingan volume rilisan yang sangat padat pada bulan Januari, serta memilih pertengahan tahun seperti bulan Juli untuk meluncurkan karya yang lebih energik. Selain itu, produser dapat memanfaatkan korelasi kuat antara energy dan loudness untuk mengoptimalkan proses mastering audio, sekaligus menggunakan pemisahan karakteristik akustik dan elektronik guna menciptakan rekomendasi musik yang lebih tepat sasaran bagi pendengar.   